In [1]:
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import matplotlib
matplotlib.use("Agg")

In [2]:
import torch
import requests
import numpy as np
import cv2
import matplotlib.pyplot as plt

from PIL import Image

from transformers import (
    AutoProcessor,
    AutoModelForZeroShotObjectDetection
)

from segment_anything import (
    sam_model_registry,
    SamPredictor
)

C:\Users\Admin\anaconda3\envs\ovs-thesis\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Using device: cuda
NVIDIA RTX PRO 4000 Blackwell


In [4]:
model_id = "IDEA-Research/grounding-dino-tiny"

processor = AutoProcessor.from_pretrained(
    model_id
)

grounding_model = AutoModelForZeroShotObjectDetection.from_pretrained(
    model_id
).to(DEVICE)

print("GroundingDINO loaded")

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 990/990 [00:00<00:00, 10420.08it/s]


GroundingDINO loaded


In [5]:
SAM_CHECKPOINT = "../checkpoints/sam_vit_h_4b8939.pth"

sam = sam_model_registry["vit_h"](
    checkpoint=SAM_CHECKPOINT
)

sam.to(device=DEVICE)

predictor = SamPredictor(sam)

print("SAM loaded")

SAM loaded


In [6]:
class GroundedSAMPipeline:

    def __init__(
        self,
        processor,
        model,
        sam_predictor,
        device
    ):

        self.processor = processor
        self.model = model
        self.predictor = sam_predictor
        self.device = device

    def detect(
        self,
        image_pil,
        text_prompt,
        box_threshold=0.25,
        text_threshold=0.25
    ):

        inputs = self.processor(
            images=image_pil,
            text=text_prompt,
            return_tensors="pt"
        ).to(self.device)

        with torch.no_grad():

            outputs = self.model(**inputs)

        results = self.processor.post_process_grounded_object_detection(
            outputs,
            inputs.input_ids,
            threshold=box_threshold,
            text_threshold=text_threshold,
            target_sizes=[image_pil.size[::-1]]
        )

        result = results[0]

        return (
            result["boxes"].cpu().numpy(),
            result["scores"].cpu().numpy(),
            result["text_labels"]
        )

    def segment(
        self,
        image_np,
        boxes
    ):

        self.predictor.set_image(image_np)

        masks_list = []

        for box in boxes:

            mask, _, _ = self.predictor.predict(
                point_coords=None,
                point_labels=None,
                box=box.astype(np.float32),
                multimask_output=False
            )

            masks_list.append(mask[0])

        return np.stack(masks_list, axis=0)

    def run(
        self,
        image_pil,
        text_prompt,
        box_threshold=0.25
    ):

        boxes, scores, labels = self.detect(
            image_pil,
            text_prompt,
            box_threshold
        )

        image_np = np.array(
            image_pil.convert("RGB")
        )

        masks = self.segment(
            image_np,
            boxes
        )

        return {
            "image_np": image_np,
            "boxes": boxes,
            "scores": scores,
            "labels": labels,
            "masks": masks
        }

In [7]:
pipeline = GroundedSAMPipeline(
    processor,
    grounding_model,
    predictor,
    DEVICE
)

print("Pipeline ready")

Pipeline ready


In [96]:
image_url = "https://farm1.staticflickr.com/99/286317888_af15d22786_z.jpg"

image_pil = Image.open(
    requests.get(
        image_url,
        stream=True
    ).raw
).convert("RGB")

print(image_pil.size)

(640, 480)


In [97]:
text_prompt = "person. laptop. fork."

result = pipeline.run(
    image_pil,
    text_prompt,
    box_threshold=0.25
)

print(f"Detected {len(result['labels'])} objects")

for label, score in zip(
    result["labels"],
    result["scores"]
):
    print(label, float(score))

Detected 35 objects
person 0.7092143297195435
laptop 0.7836759686470032
person 0.7224923968315125
person 0.562313973903656
laptop 0.7744101285934448
person 0.805746853351593
laptop 0.7358654737472534
person 0.5983185768127441
person 0.5448105335235596
person 0.6249788999557495
person 0.6072614789009094
person 0.5002124309539795
fork 0.5026368498802185
person 0.4845033586025238
person 0.3994757831096649
person 0.27229228615760803
person 0.44822144508361816
person 0.48762449622154236
person 0.4263140857219696
person 0.4047512412071228
person 0.401029109954834
person 0.30487340688705444
person 0.3839479982852936
fork 0.25684937834739685
person 0.3597414493560791
person 0.31981122493743896
person 0.3769978880882263
laptop 0.43732526898384094
person 0.3176940679550171
person 0.27529802918434143
person 0.25256770849227905
person 0.35215499997138977
person 0.2722005248069763
person 0.26806503534317017
fork 0.276998907327652


In [98]:
def compute_compactnessS_score(mask):

    mask_uint8 = mask.astype(np.uint8)

    area = np.sum(mask_uint8)

    contours, _ = cv2.findContours(
        mask_uint8,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if len(contours) == 0:
        return 0.0

    perimeter = cv2.arcLength(
        contours[0],
        True
    )

    compactness = (
        4 * np.pi * area
    ) / (perimeter ** 2 + 1e-6)

    compactness = min(
        1.0,
        compactness
    )

    return float(compactness)

In [99]:
semantic_scores = result["scores"]

structural_scores = []

for mask in result["masks"]:

    structural = compute_structural_score(
        mask
    )

    structural_scores.append(
        structural
    )

In [100]:
arbitration_scores = []

for semantic, structural in zip(
    semantic_scores,
    structural_scores
):

    arbitration = (
        0.7 * float(semantic)
        +
        0.3 * structural
    )

    arbitration_scores.append(
        arbitration
    )

In [101]:
for i, (
    label,
    semantic,
    structural,
    arbitration
) in enumerate(

    zip(
        result["labels"],
        semantic_scores,
        structural_scores,
        arbitration_scores
    )

):

    print(
        f"[{i+1}] "
        f"{label:15} | "
        f"Sem={float(semantic):.3f} | "
        f"Struct={structural:.3f} | "
        f"Final={arbitration:.3f}"
    )

[1] person          | Sem=0.709 | Struct=1.000 | Final=0.796
[2] laptop          | Sem=0.784 | Struct=0.537 | Final=0.710
[3] person          | Sem=0.722 | Struct=0.489 | Final=0.653
[4] person          | Sem=0.562 | Struct=1.000 | Final=0.694
[5] laptop          | Sem=0.774 | Struct=0.411 | Final=0.665
[6] person          | Sem=0.806 | Struct=1.000 | Final=0.864
[7] laptop          | Sem=0.736 | Struct=0.371 | Final=0.626
[8] person          | Sem=0.598 | Struct=0.731 | Final=0.638
[9] person          | Sem=0.545 | Struct=0.675 | Final=0.584
[10] person          | Sem=0.625 | Struct=0.608 | Final=0.620
[11] person          | Sem=0.607 | Struct=0.687 | Final=0.631
[12] person          | Sem=0.500 | Struct=0.646 | Final=0.544
[13] fork            | Sem=0.503 | Struct=0.200 | Final=0.412
[14] person          | Sem=0.485 | Struct=0.793 | Final=0.577
[15] person          | Sem=0.399 | Struct=0.796 | Final=0.518
[16] person          | Sem=0.272 | Struct=1.000 | Final=0.491
[17] person      

In [102]:
filtered_masks = []

for mask, arbitration in zip(
    result["masks"],
    arbitration_scores
):

    if arbitration > 0.5:

        filtered_masks.append(mask)

In [103]:
import os
import cv2

os.makedirs(
    "../outputs/day2.5",
    exist_ok=True
)

overlay = result["image_np"].copy()

colors = [
    [255, 0, 0],
    [0, 255, 0],
    [0, 0, 255],
    [255, 255, 0]
]

for i, mask in enumerate(filtered_masks):

    overlay[mask > 0] = colors[i % len(colors)]

    ys, xs = np.where(mask > 0)

    if len(xs) > 0 and len(ys) > 0:

        center_x = int(np.mean(xs))
        center_y = int(np.mean(ys))

        cv2.putText(
            overlay,
            str(i + 1),
            (center_x, center_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (255, 255, 255),
            2
        )

overlay_bgr = cv2.cvtColor(
    overlay,
    cv2.COLOR_RGB2BGR
)

cv2.imwrite(
    "../outputs/day2.5/arbitration_cafe.jpg",
    overlay_bgr
)

print("Saved visualization")

Saved visualization
